<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система способов оплаты</h2>

----

### Вариант задания № 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Разработка системы различных способов оплаты на базе класса PaymentMethod. Система предусматривает онлайн-оплату, банковский перевод и оплату наличными.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
// Интерфейс для отправки чека
public interface IReceiptable
{
    void SendReceipt(string email);
}

// Интерфейс для возврата средств
public interface IRefundable
{
    void Refund(decimal amount);
}

// ДОБАВЛЕНО: интерфейс для логирования транзакций (будет реализован явно)
public interface ITransactionLogger
{
    void LogTransaction(string info);
    string GetLastLog();
}

// Базовый класс
public class PaymentMethod : ITransactionLogger // ДОБАВЛЕНО: наследование интерфейса
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;
    private decimal _commission;

    private decimal _maxAmount;
    private bool _isActive;
    private DateTime _lastTransactionDate;

    // ДОБАВЛЕНО: новые атрибуты
    private string _transactionId;
    private int _transactionCount;
    private string _lastLog;

    public int PaymentMethodId
    {
        get { return _paymentMethodId; }
        set { _paymentMethodId = value; }
    }

    public string MethodName
    {
        get { return _methodName; }
        set { _methodName = value; }
    }

    public decimal MinAmount
    {
        get { return _minAmount; }
        set { _minAmount = value; }
    }

    public decimal Commission
    {
        get { return _commission; }
        set { _commission = value; }
    }

    public string Currency { get; set; }

    public decimal MaxAmount
    {
        get { return _maxAmount; }
        set { _maxAmount = value; }
    }

    public bool IsActive
    {
        get { return _isActive; }
        set { _isActive = value; }
    }

    public DateTime LastTransactionDate
    {
        get { return _lastTransactionDate; }
        set { _lastTransactionDate = value; }
    }

    // ДОБАВЛЕНО: новые свойства
    public string TransactionId
    {
        get { return _transactionId; }
        set { _transactionId = value; }
    }

    public int TransactionCount
    {
        get { return _transactionCount; }
        set { _transactionCount = value; }
    }

    public PaymentMethod(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, decimal maxAmount = 1000000m)
    {
        PaymentMethodId = paymentMethodId;
        MethodName = methodName;
        MinAmount = minAmount;
        Commission = commission;
        Currency = "RUB";
        MaxAmount = maxAmount;
        IsActive = true;
        _transactionCount = 0;
    }

    public bool CheckMinimumAmount(decimal amount)
    {
        return amount >= MinAmount;
    }

    public bool CheckMaximumAmount(decimal amount)
    {
        return amount <= MaxAmount;
    }

    public decimal CalculateCommission(decimal amount)
    {
        return amount * (Commission / 100);
    }

    public void CancelPayment()
    {
        Console.WriteLine($"Платеж по способу '{MethodName}' отменен.");
    }

    // ДОБАВЛЕНО: новый метод генерации ID транзакции
    public void GenerateTransactionId()
    {
        TransactionId = $"{PaymentMethodId}-{DateTime.Now.Ticks}";
        Console.WriteLine($"Сгенерирован ID транзакции: {TransactionId}");
    }

    public virtual void ProcessPayment(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine($"Способ оплаты '{MethodName}' временно неактивен.");
            return;
        }

        if (!CheckMaximumAmount(amount))
        {
            Console.WriteLine(
                $"Ошибка: сумма {amount} {Currency} превышает максимальную ({MaxAmount} {Currency}).");
            return;
        }

        if (CheckMinimumAmount(amount))
        {
            decimal commission = CalculateCommission(amount);
            LastTransactionDate = DateTime.Now;
            _transactionCount++; // ДОБАВЛЕНО: счётчик транзакций
            GenerateTransactionId(); // ДОБАВЛЕНО

            Console.WriteLine(
                $"Успешно: платеж на сумму {amount} {Currency} " +
                $"(комиссия: {commission} {Currency}) проведен через '{MethodName}'.");

            ((ITransactionLogger)this).LogTransaction($"Платеж {amount} {Currency} через {MethodName}"); // ДОБАВЛЕНО
        }
        else
        {
            Console.WriteLine(
                $"Ошибка: сумма {amount} {Currency} меньше минимальной " +
                $"({MinAmount} {Currency}) для '{MethodName}'.");
        }
    }

    public void ProcessPayment(decimal amount, string comment)
    {
        ProcessPayment(amount);
        Console.WriteLine($"Комментарий к платежу: {comment}");
    }

    public virtual void GetPaymentDetails()
    {
        Console.WriteLine(
            $"ID способа: {PaymentMethodId} | Название: {MethodName} | " +
            $"Мин. сумма: {MinAmount} {Currency} | Макс. сумма: {MaxAmount} {Currency} | " +
            $"Комиссия: {Commission}% | Активен: {IsActive} | Транзакций: {TransactionCount}"); // ДОБАВЛЕНО TransactionCount
    }

    // ДОБАВЛЕНО: явная реализация интерфейса ITransactionLogger
    void ITransactionLogger.LogTransaction(string info)
    {
        _lastLog = $"[{DateTime.Now:HH:mm:ss}] {info}";
        Console.WriteLine($"Лог записан: {_lastLog}");
    }

    string ITransactionLogger.GetLastLog()
    {
        return _lastLog ?? "Логов пока нет.";
    }
}

// Онлайн-оплата
public class OnlinePayment : PaymentMethod, IReceiptable
{
    private string _paymentUrl;

    public string PaymentUrl
    {
        get { return _paymentUrl; }
        set { _paymentUrl = value; }
    }

    public OnlinePayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string paymentUrl)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        PaymentUrl = paymentUrl;
    }

    public void SendReceipt(string email)
    {
        Console.WriteLine($"Электронный чек отправлен на почту: {email}");
    }

    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"URL для оплаты: {PaymentUrl}");
    }
}

// Банковский перевод
public class BankTransfer : PaymentMethod
{
    private string _bankData;

    // ДОБАВЛЕНО: новый атрибут
    private string _bic;

    public string BankData
    {
        get { return _bankData; }
        set { _bankData = value; }
    }

    // ДОБАВЛЕНО: новое свойство
    public string Bic
    {
        get { return _bic; }
        set { _bic = value; }
    }

    public BankTransfer(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string bankData, string bic = "044525225")
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        BankData = bankData;
        Bic = bic; // ДОБАВЛЕНО
    }

    public override void ProcessPayment(decimal amount)
    {
        base.ProcessPayment(amount);
        Console.WriteLine($"Перевод отправлен по реквизитам: {BankData}, БИК: {Bic}");
    }

    // ДОБАВЛЕНО: новый метод
    public void VerifyBankDetails()
    {
        Console.WriteLine($"Проверка реквизитов: {BankData} / БИК {Bic} — ОК.");
    }
}

// Оплата наличными
public class CashPayment : PaymentMethod
{
    private string _cashPickupPoint;

    // ДОБАВЛЕНО: новый атрибут
    private TimeSpan _workingHours;

    public string CashPickupPoint
    {
        get { return _cashPickupPoint; }
        set { _cashPickupPoint = value; }
    }

    // ДОБАВЛЕНО: новое свойство
    public TimeSpan WorkingHours
    {
        get { return _workingHours; }
        set { _workingHours = value; }
    }

    public CashPayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string cashPickupPoint)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        CashPickupPoint = cashPickupPoint;
        WorkingHours = new TimeSpan(10, 0, 0); // ДОБАВЛЕНО: 10 часов работы по умолчанию
    }

    public override void ProcessPayment(decimal amount)
    {
        base.ProcessPayment(amount);
        Console.WriteLine($"Ожидайте оплату наличными в точке: {CashPickupPoint}");
    }

    // ДОБАВЛЕНО: новый метод
    public void CheckAvailability()
    {
        Console.WriteLine($"Точка {CashPickupPoint} работает {WorkingHours.Hours} часов в день.");
    }
}

// Оплата банковской картой
public class CardPayment : OnlinePayment, IRefundable
{
    public string CardNumber { get; set; }
    public string CardHolder { get; set; }

    // ДОБАВЛЕНО: новый атрибут
    public DateTime CardExpiry { get; set; }

    public CardPayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string paymentUrl,
        string cardNumber, string cardHolder, DateTime? cardExpiry = null)
        : base(paymentMethodId, methodName, minAmount, commission, paymentUrl)
    {
        CardNumber = cardNumber;
        CardHolder = cardHolder;
        CardExpiry = cardExpiry ?? DateTime.Now.AddYears(3); // ДОБАВЛЕНО
    }

    public void Refund(decimal amount)
    {
        Console.WriteLine(
            $"Оформлен возврат {amount} {Currency} на карту {CardNumber} ({CardHolder})");
    }

    public void Refund(decimal amount, string reason)
    {
        Refund(amount);
        Console.WriteLine($"Причина возврата: {reason}");
    }

    public override void ProcessPayment(decimal amount)
    {
        base.ProcessPayment(amount);
        Console.WriteLine($"Списание с карты {CardNumber}, держатель: {CardHolder}");
    }

    // ДОБАВЛЕНО: новый метод
    public bool IsCardExpired()
    {
        return DateTime.Now > CardExpiry;
    }
}

// Generic класс для хранения и обработки коллекции способов оплаты
public class PaymentCollection<T> where T : PaymentMethod
{
    private List<T> _payments = new List<T>();

    public void Add(T payment)
    {
        _payments.Add(payment);
    }

    public void ProcessAll(decimal amount)
    {
        foreach (var payment in _payments)
        {
            payment.ProcessPayment(amount);
        }
    }

    public void ShowAllDetails()
    {
        foreach (var payment in _payments)
        {
            payment.GetPaymentDetails();
        }
    }
}

// Класс заказа
public class Order
{
    public int OrderId { get; set; }
    public decimal TotalAmount { get; set; }

    public Order(int orderId, decimal totalAmount)
    {
        OrderId = orderId;
        TotalAmount = totalAmount;
    }

    public void Pay(PaymentMethod paymentMethod)
    {
        Console.WriteLine(
            $"\nПопытка оплаты заказа №{OrderId} на сумму {TotalAmount} руб.");

        paymentMethod.ProcessPayment(TotalAmount);
    }
}

// ДОБАВЛЕНО: класс управления зависимостями (DI) — не создаёт способ оплаты сам,
// а получает его извне через конструктор
public class PaymentProcessorService
{
    private readonly PaymentMethod _paymentMethod;

    public PaymentProcessorService(PaymentMethod paymentMethod)
    {
        _paymentMethod = paymentMethod;
    }

    public void Execute(decimal amount)
    {
        _paymentMethod.GetPaymentDetails();
        _paymentMethod.ProcessPayment(amount);

        ITransactionLogger logger = _paymentMethod; // приведение к интерфейсу для доступа к явной реализации
        Console.WriteLine($"Последний лог: {logger.GetLastLog()}");
    }
}


// --- Создание объектов ---
BankTransfer bank = new BankTransfer(
    1, "Банковский перевод", 500, 1.5m,
    "ПАО Сбербанк, Р/С 407...");

CashPayment cash = new CashPayment(
    2, "Наличные", 50, 0m,
    "Касса №1 (ул. Мира, 10)");

CardPayment card = new CardPayment(
    3, "Банковская карта онлайн", 100, 2.0m,
    "https://securepay.ru",
    "2200-****-****-1234",
    "Иван Иванов");

// --- Вывод информации ---
Console.WriteLine("--- Информация о способах оплаты ---");
bank.GetPaymentDetails();
Console.WriteLine();
cash.GetPaymentDetails();
Console.WriteLine();
card.GetPaymentDetails();
Console.WriteLine();

// --- Проверка новых методов ---
Console.WriteLine("--- Проверка новых методов ---");
bank.VerifyBankDetails();
cash.CheckAvailability();
Console.WriteLine($"Карта просрочена? {card.IsCardExpired()}");
Console.WriteLine();

// --- Проверка перегрузки методов ---
Console.WriteLine("--- Проверка перегрузки методов ---");
card.ProcessPayment(1500);
card.ProcessPayment(1500, "Оплата за заказ №55");
card.Refund(500);
card.Refund(500, "Товар не подошел");

// --- Проверка явной реализации интерфейса ---
Console.WriteLine("\n--- Проверка явной реализации ITransactionLogger ---");
ITransactionLogger logger = card; // доступ только через интерфейс
logger.LogTransaction("Ручная проверка лога");
Console.WriteLine(logger.GetLastLog());
// card.LogTransaction(...); // ТАК НЕЛЬЗЯ — ошибка компиляции, метод скрыт

// --- Работа Generic класса ---
Console.WriteLine("\n--- Работа Generic класса PaymentCollection ---");
PaymentCollection<PaymentMethod> collection = new PaymentCollection<PaymentMethod>();
collection.Add(bank);
collection.Add(cash);
collection.Add(card);

collection.ShowAllDetails();
Console.WriteLine();
collection.ProcessAll(600);

// --- Взаимодействие с классом Order ---
Console.WriteLine("\n--- Взаимодействие с классом Order ---");
Order order1 = new Order(101, 1000);
Order order2 = new Order(102, 30);

order1.Pay(card);
order2.Pay(card);

// --- ДОБАВЛЕНО: проверка управления зависимостями (DI) ---
Console.WriteLine("\n--- Проверка PaymentProcessorService (DI) ---");
PaymentProcessorService serviceWithCard = new PaymentProcessorService(card);
serviceWithCard.Execute(2000);

PaymentProcessorService serviceWithBank = new PaymentProcessorService(bank);
serviceWithBank.Execute(700);

--- Информация о способах оплаты ---
ID способа: 1 | Название: Банковский перевод | Мин. сумма: 500 RUB | Макс. сумма: 1000000 RUB | Комиссия: 1.5% | Активен: True | Транзакций: 0

ID способа: 2 | Название: Наличные | Мин. сумма: 50 RUB | Макс. сумма: 1000000 RUB | Комиссия: 0% | Активен: True | Транзакций: 0

ID способа: 3 | Название: Банковская карта онлайн | Мин. сумма: 100 RUB | Макс. сумма: 1000000 RUB | Комиссия: 2.0% | Активен: True | Транзакций: 0
URL для оплаты: https://securepay.ru

--- Проверка новых методов ---
Проверка реквизитов: ПАО Сбербанк, Р/С 407... / БИК 044525225 — ОК.
Точка Касса №1 (ул. Мира, 10) работает 10 часов в день.
Карта просрочена? False

--- Проверка перегрузки методов ---
Сгенерирован ID транзакции: 3-639285359804419779
Успешно: платеж на сумму 1500 RUB (комиссия: 30.00 RUB) проведен через 'Банковская карта онлайн'.
Лог записан: [14:39:40] Платеж 1500 RUB через Банковская карта онлайн
Списание с карты 2200-****-****-1234, держатель: Иван Иванов
Сгенерир